# 02 — Refinamento do Qwen3-0.6B com raciocínio (LoRA/QLoRA)

Este é o caminho **recomendado** para o laboratório:

**Qwen3-0.6B (HF) → SFT com LoRA/QLoRA → adapter PEFT → merge → GGUF no próximo notebook.**

O Qwen3-0.6B é pequeno, aberto sob Apache-2.0 e possui modo thinking/non-thinking. O dataset de demonstração usa respostas estruturadas com `<think>...</think>` seguidas da resposta final.

> Para um projeto real, use raciocínios corretos e verificáveis. Dados de baixa qualidade ensinam o modelo a reproduzir erros com mais confiança.

In [ ]:
%pip install -U "transformers>=4.51.0" "datasets>=3" "accelerate>=1" "peft>=0.14" "bitsandbytes>=0.45" safetensors

In [ ]:
from pathlib import Path
import json, os, random
import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = 'Qwen/Qwen3-0.6B'
ADAPTER_DIR = Path('artifacts/qwen3-0.6b-raciocinio-lora')
MERGED_DIR = Path('artifacts/qwen3-0.6b-raciocinio-merged')
DATA_FILE = Path('data/raciocinio_sft.jsonl')
MAX_LENGTH = 768   # use 512 com pouca VRAM; aumente conforme o dataset/hardware

## 1. Dataset didático

Formato: uma linha JSON por conversa. Você pode substituir o arquivo por seu dataset mantendo a coluna `messages`.

Nos exemplos abaixo o bloco de raciocínio é curto de propósito. Em tarefas matemáticas, prefira etapas verificáveis; em tarefas gerais, muitas vezes basta treinar uma justificativa concisa e a resposta final.

In [ ]:
examples = [
  {"messages": [
    {"role":"user","content":"/think Se uma caixa tem 6 fileiras com 8 peças em cada fileira, quantas peças há ao todo?"},
    {"role":"assistant","content":"<think>Há 6 grupos de 8 peças. Multiplico 6 por 8 e obtenho 48.</think>\n\n48 peças."}
  ]},
  {"messages": [
    {"role":"user","content":"/think Qual é o próximo número da sequência 2, 4, 8, 16, ?"},
    {"role":"assistant","content":"<think>Cada termo é o dobro do anterior; então 16 vezes 2 é 32.</think>\n\n32."}
  ]},
  {"messages": [
    {"role":"user","content":"/think Um produto de R$ 120 recebeu desconto de 25%. Qual é o preço final?"},
    {"role":"assistant","content":"<think>25% de 120 é 30. Subtraindo 30 de 120, o preço final é 90.</think>\n\nR$ 90,00."}
  ]},
  {"messages": [
    {"role":"user","content":"/think Em Python, por que usar um conjunto (set) para testar pertencimento em muitos itens?"},
    {"role":"assistant","content":"<think>Sets usam tabela hash na implementação comum, então a busca média por pertencimento é constante, enquanto uma lista exige varredura linear.</think>\n\nPorque `x in conjunto` costuma ser O(1) em média, contra O(n) para uma lista."}
  ]},
  {"messages": [
    {"role":"user","content":"/think Se A implica B e A é verdadeiro, o que podemos concluir?"},
    {"role":"assistant","content":"<think>Pelo modus ponens, de A→B e A concluímos B.</think>\n\nB é verdadeiro."}
  ]},
  {"messages": [
    {"role":"user","content":"/think Uma API respondeu 500 em 3 de 1000 requisições. Qual a taxa percentual de erro?"},
    {"role":"assistant","content":"<think>A fração é 3/1000 = 0,003. Multiplicando por 100, resulta 0,3%.</think>\n\n0,3%."}
  ]},
  {"messages": [
    {"role":"user","content":"/no_think Responda apenas com o resultado: 17 + 28."},
    {"role":"assistant","content":"45"}
  ]},
  {"messages": [
    {"role":"user","content":"/no_think Qual estrutura JSON representa uma lista vazia?"},
    {"role":"assistant","content":"[]"}
  ]},
]

DATA_FILE.parent.mkdir(exist_ok=True)
with DATA_FILE.open('w', encoding='utf-8') as f:
    # Repete para smoke test. Troque por dados reais antes de avaliar qualidade.
    for _ in range(12):
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')
print('Exemplos gravados:', sum(1 for _ in DATA_FILE.open(encoding='utf-8')))

## 2. Carregar tokenizer e modelo

Se houver CUDA, o notebook usa 4-bit NF4 (**QLoRA**). Sem CUDA, cai para LoRA em precisão normal para manter o código executável, embora muito mais lento.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

cuda = torch.cuda.is_available()
compute_dtype = torch.bfloat16 if cuda and torch.cuda.is_bf16_supported() else (torch.float16 if cuda else torch.float32)

print('CUDA:', cuda, '| compute dtype:', compute_dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if cuda:
    qconfig = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=qconfig,
        device_map='auto',
        torch_dtype=compute_dtype,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)

model.config.use_cache = False
print('Modelo carregado.')

## 3. Inserir LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if cuda:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model.gradient_checkpointing_enable()

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 4. Renderizar o chat template e tokenizar

O parâmetro `enable_thinking=True` preserva o comportamento híbrido do Qwen3. Os comandos `/think` e `/no_think` presentes nos exemplos também ensinam alternância de comportamento.

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files=str(DATA_FILE), split='train').train_test_split(test_size=0.1, seed=SEED)

def render(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=True,
    )
    return {'text': text}

ds = ds.map(render)
print(ds['train'][0]['text'])

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )

tok_ds = ds.map(tokenize, batched=True, remove_columns=ds['train'].column_names)
print(tok_ds)

## 5. SFT

Este caderno usa `Trainer` para reduzir dependência de APIs específicas do TRL. O `DataCollatorForLanguageModeling(mlm=False)` mascara apenas padding e treina previsão causal sobre a conversa completa.

Para um curso avançado, o próximo exercício é mascarar também os tokens de `system/user` e calcular loss somente sobre tokens do `assistant`.

In [ ]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    # `warmup_ratio` foi removido no Transformers 5; restou apenas `warmup_steps`.
    # No smoke test (~24 passos de otimizador), 5 passos de aquecimento ~ 5% do treino.
    warmup_steps=5,
    lr_scheduler_type='cosine',
    logging_steps=2,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
    seed=SEED,
    gradient_checkpointing=True,
    bf16=cuda and torch.cuda.is_bf16_supported(),
    fp16=cuda and not torch.cuda.is_bf16_supported(),
    # `paged_adamw_8bit` exige memória unificada (UVM), indisponível no Windows.
    # `adamw_bnb_8bit` economiza estado do otimizador sem paginação.
    optim='adamw_bnb_8bit' if cuda else 'adamw_torch',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_ds['train'],
    eval_dataset=tok_ds['test'],
    data_collator=collator,
)
trainer.train()


In [ ]:
trainer.evaluate()

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Adapter LoRA salvo em:', ADAPTER_DIR)

## 6. Teste rápido do adapter

In [ ]:
model.eval()
# O treino deixou o gradient checkpointing ativo, o que força use_cache=False e
# deixa a geração lenta. Desligue antes de gerar.
if getattr(model, "gradient_checkpointing_enable", None):
    model.gradient_checkpointing_disable()
model.config.use_cache = True

messages = [{"role":"user", "content":"/think Uma máquina processa 45 itens por minuto. Quantos itens processa em 8 minutos?"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
new_tokens = out[0, inputs['input_ids'].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


## 7. Merge do LoRA para conversão GGUF

Recarregamos o modelo base sem 4-bit e aplicamos o adapter. Isso produz um diretório Hugging Face convencional que o conversor do `llama.cpp` pode ler.

In [ ]:
# Libere a cópia QLoRA antes do merge para economizar VRAM/RAM.
import gc
try:
    del trainer
except Exception:
    pass
try:
    del model
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_config = ADAPTER_DIR / 'adapter_config.json'
if not adapter_config.is_file():
    raise FileNotFoundError(
        f'Adaptador ausente em {ADAPTER_DIR}. Execute as células 5 (SFT) e 6 (salvar) antes do merge.'
    )

# CPU é suficiente para 0.6B e evita pressão na VRAM.
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32, device_map='cpu')
merged = PeftModel.from_pretrained(base, str(ADAPTER_DIR)).merge_and_unload()

MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='2GB')
AutoTokenizer.from_pretrained(MODEL_NAME).save_pretrained(MERGED_DIR)
print('Modelo merged salvo em:', MERGED_DIR)


## Próximo passo

Abra `03_exportar_e_testar_gguf.ipynb` para converter `artifacts/qwen3-0.6b-raciocinio-merged` em GGUF e quantizar para Q4_K_M/Q8_0.